In [1]:
import pandas as pd
import numpy as np
import pickle
import sys, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

# ── Load all pipeline outputs ────────────────────────────────────────────────
mapping  = pd.read_csv(f'{OUT_PATH}archetype_mapping.csv')
keys_df  = pd.read_csv(f'{OUT_PATH}archetype_keys.csv')
abt      = pd.read_csv(f'{OUT_PATH}07_analytical_base_table.csv')
monthly  = pd.read_csv(f'{OUT_PATH}07_archetype_monthly.csv')

abt['sale_date'] = pd.to_datetime(abt['sale_date'])
abt['year']      = abt['sale_date'].dt.year
abt['month']     = abt['sale_date'].dt.month

bts = pd.read_csv(f'{OUT_PATH}02_fine_bucket_ts.csv')
bts['sale_date'] = pd.to_datetime(bts['sale_date'])
bts['year']      = bts['sale_date'].dt.year
bts['month']     = bts['sale_date'].dt.month

with open(f'{OUT_PATH}03_segment_pivots.pkl', 'rb') as f:
    segment_pivots = pickle.load(f)

print(f"Channel          : {CHANNEL}")
print(f"Output path      : {OUT_PATH}")
print(f"MAX_K cap        : {MAX_K}")
print(f"archetype_mapping: {mapping.shape}")
print(f"archetype_keys   : {keys_df.shape}")
print(f"ABT              : {abt.shape}")
print(f"archetype_monthly: {monthly.shape}")
print(f"segment_pivots   : {len(segment_pivots)} segments")

# Years present in data
ALL_YEARS = sorted(abt['year'].unique())
print(f"Years in data    : {ALL_YEARS}")


Channel          : TT
Output path      : data/outputs/TT/
MAX_K cap        : 8
archetype_mapping: (2142, 9)
archetype_keys   : (96, 9)
ABT              : (43905, 18)
archetype_monthly: (10011, 11)
segment_pivots   : 14 segments
Years in data    : [np.int32(2024), np.int32(2025)]


In [2]:
# ════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — per-portal and consolidated
# ════════════════════════════════════════════════════════════════════

def portal_to_str(portal):
    """Filesystem-safe label for a portal (handles integer portals like TT)."""
    if portal in PORTAL_ABBREV:
        return str(PORTAL_ABBREV[portal])
    return str(portal).replace('/', '_').replace(' ', '_').replace('\\', '_')


def get_portals(div, size, keys_df_in):
    """Sorted list of portals for (Division, Size) in archetype_keys."""
    return sorted(
        keys_df_in[(keys_df_in['Division'] == div) & (keys_df_in['Size'] == size)]['Portal'].unique(),
        key=str
    )


# ── Per-portal helpers ──────────────────────────────────────────────────────────────────────────

def build_summary_portal(div, portal, size, mapping_df, keys_df_in, abt_in):
    """Summary table for a single (Division, Portal, Size) segment."""
    seg_map  = mapping_df[(mapping_df['Division']==div) & (mapping_df['Portal']==portal) & (mapping_df['Size']==size)].copy()
    seg_keys = keys_df_in[(keys_df_in['Division']==div) & (keys_df_in['Portal']==portal) & (keys_df_in['Size']==size)].copy()
    seg_abt  = abt_in[(abt_in['Division']==div) & (abt_in['Portal']==portal) & (abt_in['Size']==size)].copy()

    if seg_keys.empty:
        return pd.DataFrame()

    rows = []
    total_seg_qty = seg_map['total_qty'].sum()

    for _, krow in seg_keys.sort_values('New_Bucket').iterrows():
        nb  = krow['New_Bucket']
        key = krow['archetype_key']
        nb_map = seg_map[seg_map['New_Bucket'] == nb]
        nb_abt = seg_abt[seg_abt['New_Bucket'] == nb]

        total_qty    = nb_map['total_qty'].sum()
        total_ns     = nb_abt['net_sales'].sum() if not nb_abt.empty else 0
        total_asp    = round(total_ns / total_qty, 2) if total_qty > 0 else 0
        vol_pct      = round(100 * total_qty / total_seg_qty, 2) if total_seg_qty > 0 else 0
        fine_buckets = int((nb_map['total_qty'] > 0).sum())

        row = {
            'Division':      div,
            'Portal':        portal,
            'Size':          size,
            'New_Bucket':    int(nb),
            'archetype_key': key,
            'price_min':     int(krow['price_range_min']),
            'price_max':     int(krow['price_range_max']),
            'fine_buckets':  fine_buckets,
            'total_qty':     int(total_qty),
            'net_sales':     round(total_ns, 2),
            'ASP':           total_asp,
            'vol_pct':       vol_pct,
        }
        for yr in ALL_YEARS:
            yr_abt = nb_abt[nb_abt['year'] == yr]
            q = int(yr_abt['qty'].sum()) if not yr_abt.empty else 0
            n = round(yr_abt['net_sales'].sum(), 2) if not yr_abt.empty else 0
            a = round(n / q, 2) if q > 0 else 0
            row[f'qty_{yr}'] = q
            row[f'ns_{yr}']  = n
            row[f'ASP_{yr}'] = a
        rows.append(row)

    return pd.DataFrame(rows)


def build_detail_portal(div, portal, size, abt_in):
    """Monthly detail for a single (Division, Portal, Size) segment."""
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Portal']==portal) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    grp = (seg.groupby(['New_Bucket', 'archetype_key', 'bucket_min', 'year', 'month'])
              .agg(qty=('qty', 'sum'), net_sales=('net_sales', 'sum'))
              .reset_index())
    grp['ASP']     = (grp['net_sales'] / grp['qty'].replace(0, np.nan)).round(2)
    seg_total_qty  = grp['qty'].sum()
    grp['vol_pct'] = (grp['qty'] / seg_total_qty * 100).round(4) if seg_total_qty > 0 else 0
    return grp.sort_values(['New_Bucket', 'bucket_min', 'year', 'month'])


def build_trend_pivot_portal(div, portal, size, abt_in):
    """Monthly trend pivot for a single (Division, Portal, Size). No duplicate-label risk."""
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Portal']==portal) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    seg_grp = (seg.groupby(['New_Bucket', 'archetype_key', 'year', 'month'])
                  .agg(qty=('qty', 'sum'), net_sales=('net_sales', 'sum'))
                  .reset_index())

    nb_keys = seg_grp[['New_Bucket', 'archetype_key']].drop_duplicates().sort_values('New_Bucket')
    rows = []
    for _, nkrow in nb_keys.iterrows():
        nb  = nkrow['New_Bucket']
        key = nkrow['archetype_key']
        nb_seg = seg_grp[seg_grp['New_Bucket'] == nb]
        for m in range(1, 13):
            row = {'New_Bucket': int(nb), 'archetype_key': key, 'month': m}
            for yr in ALL_YEARS:
                yr_m = nb_seg[(nb_seg['year']==yr) & (nb_seg['month']==m)]
                q = int(yr_m['qty'].sum())
                n = round(yr_m['net_sales'].sum(), 2)
                a = round(n / q, 2) if q > 0 else np.nan
                seg_yr_m  = seg_grp[(seg_grp['year']==yr) & (seg_grp['month']==m)]
                seg_m_qty = seg_yr_m['qty'].sum()
                pct = round(100 * q / seg_m_qty, 4) if seg_m_qty > 0 else 0
                row[f'qty_{yr}']       = q
                row[f'net_sales_{yr}'] = n
                row[f'ASP_{yr}']       = a
                row[f'pct_share_{yr}'] = pct
            rows.append(row)
    return pd.DataFrame(rows)


def build_pivot_ready_portal(div, portal, size, abt_in):
    """Wide-format qty pivot for a single (Division, Portal, Size). Cols: YYYY-MM."""
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Portal']==portal) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    grp = (seg.groupby(['New_Bucket', 'archetype_key', 'year', 'month'])
              .agg(qty=('qty', 'sum'))
              .reset_index())
    pivot = grp.pivot_table(index='New_Bucket',
                            columns=['year', 'month'],
                            values='qty',
                            aggfunc='sum',
                            fill_value=0)
    pivot.columns = [f'{y}-{m:02d}' for y, m in pivot.columns]
    nb_to_key = grp[['New_Bucket', 'archetype_key']].drop_duplicates().set_index('New_Bucket')['archetype_key']
    pivot.insert(0, 'archetype_key', pivot.index.map(nb_to_key))
    date_cols = [c for c in pivot.columns if c not in ['archetype_key']]
    pivot['TOTAL_QTY'] = pivot[date_cols].sum(axis=1)
    total = pivot['TOTAL_QTY'].sum()
    pivot['VOL_PCT'] = (100 * pivot['TOTAL_QTY'] / total).round(1) if total > 0 else 0.0
    return pivot


# ── Consolidated helpers (Division × Size, portals become columns) ────────────────────────

def build_summary_consolidated(div, size, mapping_df, keys_df_in, abt_in):
    """
    Wide-format summary for (Division, Size).
    Row grain: New_Bucket ordinal rank. Portal metrics become columns:
    archetype_key_{p}, price_min_{p}, price_max_{p}, qty_{p}, ns_{p}, ASP_{p},
    vol_pct_{p}, qty_{yr}_{p}, ns_{yr}_{p}, ASP_{yr}_{p}.
    Missing portals for a given NB rank are filled with NaN/0.
    """
    portals = get_portals(div, size, keys_df_in)
    if not portals:
        return pd.DataFrame()

    portal_summaries = {}
    all_nbs = set()
    for portal in portals:
        ps = build_summary_portal(div, portal, size, mapping_df, keys_df_in, abt_in)
        if not ps.empty:
            ps['New_Bucket'] = ps['New_Bucket'].astype(int)
            portal_summaries[portal] = ps.set_index('New_Bucket')
            all_nbs.update(ps['New_Bucket'].tolist())

    if not portal_summaries:
        return pd.DataFrame()

    rows = []
    for nb in sorted(all_nbs):
        row = {'Division': div, 'Size': size, 'New_Bucket': nb}
        for portal in portals:
            ps = portal_summaries.get(portal)
            p  = portal_to_str(portal)
            if ps is not None and nb in ps.index:
                r = ps.loc[nb]
                if isinstance(r, pd.DataFrame):
                    r = r.iloc[0]
                row[f'archetype_key_{p}'] = r['archetype_key']
                row[f'price_min_{p}']     = r['price_min']
                row[f'price_max_{p}']     = r['price_max']
                row[f'qty_{p}']           = r['total_qty']
                row[f'ns_{p}']            = r['net_sales']
                row[f'ASP_{p}']           = r['ASP']
                row[f'vol_pct_{p}']       = r['vol_pct']
                for yr in ALL_YEARS:
                    row[f'qty_{yr}_{p}']  = r.get(f'qty_{yr}', 0)
                    row[f'ns_{yr}_{p}']   = r.get(f'ns_{yr}', 0)
                    row[f'ASP_{yr}_{p}']  = r.get(f'ASP_{yr}', np.nan)
            else:
                row[f'archetype_key_{p}'] = np.nan
                row[f'price_min_{p}']     = np.nan
                row[f'price_max_{p}']     = np.nan
                row[f'qty_{p}']           = 0
                row[f'ns_{p}']            = 0
                row[f'ASP_{p}']           = np.nan
                row[f'vol_pct_{p}']       = np.nan
                for yr in ALL_YEARS:
                    row[f'qty_{yr}_{p}']  = 0
                    row[f'ns_{yr}_{p}']   = 0
                    row[f'ASP_{yr}_{p}']  = np.nan
        rows.append(row)

    return pd.DataFrame(rows)


def build_detail_consolidated(div, size, abt_in):
    """
    Monthly detail for (Division, Size) keeping Portal as a column.
    groupby includes Portal so each portal's NB/archetype stays distinct.
    """
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    grp = (seg.groupby(['Portal', 'New_Bucket', 'archetype_key', 'bucket_min', 'year', 'month'])
              .agg(qty=('qty', 'sum'), net_sales=('net_sales', 'sum'))
              .reset_index())
    grp['ASP']     = (grp['net_sales'] / grp['qty'].replace(0, np.nan)).round(2)
    seg_total_qty  = grp['qty'].sum()
    grp['vol_pct'] = (grp['qty'] / seg_total_qty * 100).round(4) if seg_total_qty > 0 else 0
    return grp.sort_values(['Portal', 'New_Bucket', 'bucket_min', 'year', 'month'])


def build_trend_pivot_consolidated(div, size, abt_in):
    """
    Monthly trend pivot aggregated across all portals for (Division, Size).
    Fix: groupby(['New_Bucket','year','month']) collapses portals so there are
    never duplicate labels before any pivot/reindex operation.
    """
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    seg_grp = (seg.groupby(['New_Bucket', 'year', 'month'])
                  .agg(qty=('qty', 'sum'), net_sales=('net_sales', 'sum'))
                  .reset_index())

    all_nbs = sorted(seg_grp['New_Bucket'].unique())
    rows = []
    for nb in all_nbs:
        nb_seg = seg_grp[seg_grp['New_Bucket'] == nb]
        for m in range(1, 13):
            row = {'Division': div, 'Size': size, 'New_Bucket': int(nb), 'month': m}
            for yr in ALL_YEARS:
                yr_m = nb_seg[(nb_seg['year']==yr) & (nb_seg['month']==m)]
                q = int(yr_m['qty'].sum())
                n = round(yr_m['net_sales'].sum(), 2)
                a = round(n / q, 2) if q > 0 else np.nan
                seg_yr_m  = seg_grp[(seg_grp['year']==yr) & (seg_grp['month']==m)]
                seg_m_qty = seg_yr_m['qty'].sum()
                pct = round(100 * q / seg_m_qty, 4) if seg_m_qty > 0 else 0
                row[f'qty_{yr}']       = q
                row[f'net_sales_{yr}'] = n
                row[f'ASP_{yr}']       = a
                row[f'pct_share_{yr}'] = pct
            rows.append(row)

    return pd.DataFrame(rows)


def build_pivot_ready_consolidated(div, size, abt_in):
    """
    Wide-format qty pivot aggregated across portals for (Division, Size).
    Cols: YYYY-MM. Rows: New_Bucket.
    """
    seg = abt_in[(abt_in['Division']==div) & (abt_in['Size']==size)].copy()
    if seg.empty:
        return pd.DataFrame()

    grp = (seg.groupby(['New_Bucket', 'year', 'month'])
              .agg(qty=('qty', 'sum'))
              .reset_index())
    pivot = grp.pivot_table(index='New_Bucket',
                            columns=['year', 'month'],
                            values='qty',
                            aggfunc='sum',
                            fill_value=0)
    pivot.columns = [f'{y}-{m:02d}' for y, m in pivot.columns]
    date_cols = list(pivot.columns)
    pivot['TOTAL_QTY'] = pivot[date_cols].sum(axis=1)
    total = pivot['TOTAL_QTY'].sum()
    pivot['VOL_PCT'] = (100 * pivot['TOTAL_QTY'] / total).round(1) if total > 0 else 0.0
    pivot.insert(0, 'Division', div)
    pivot.insert(1, 'Size', size)
    return pivot


print('Helper functions defined.')


Helper functions defined.


In [3]:
# ════════════════════════════════════════════════════════════════════
# PNG FUNCTIONS
# ════════════════════════════════════════════════════════════════════

def make_full_analysis_png_portal(div, portal, size, summary_df, abt_in, out_path):
    """
    4-panel analysis PNG for a single (Division, Portal, Size) segment.
    Panels: volume bar | YoY qty | monthly % trend | summary table.
    """
    if summary_df.empty:
        return

    portal_str = portal_to_str(portal)
    seg_label  = f"{div} / {portal_str} / {size}"
    n_buckets  = len(summary_df)

    fig = plt.figure(figsize=(20, 14))
    fig.suptitle(f"{CHANNEL} | {seg_label} | K={n_buckets} archetypes",
                 fontsize=15, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_buckets, 10)))

    # Panel 1: Volume % bar
    nb_labels = [f"NB{int(r['New_Bucket'])}" for _, r in summary_df.iterrows()]
    vol_pcts  = summary_df['vol_pct'].values
    bars = ax1.barh(nb_labels, vol_pcts, color=colors[:n_buckets])
    ax1.set_xlabel('% of portal segment volume')
    ax1.set_title('Volume share per archetype', fontweight='bold')
    for bar, val in zip(bars, vol_pcts):
        ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}%', va='center', fontsize=8)
    ax1.set_xlim(0, (max(vol_pcts) * 1.2) if len(vol_pcts) else 1)

    # Panel 2: YoY qty
    qty_cols = [c for c in summary_df.columns if c.startswith('qty_')]
    x = np.arange(n_buckets)
    width = 0.8 / max(len(qty_cols), 1)
    for j, qc in enumerate(qty_cols):
        yr = qc.replace('qty_', '')
        ax2.bar(x + j * width - (len(qty_cols) - 1) * width / 2,
                summary_df[qc].values, width, label=yr, alpha=0.8)
    ax2.set_xticks(x)
    ax2.set_xticklabels(nb_labels, fontsize=8)
    ax2.set_ylabel('Units (qty)')
    ax2.set_title('Year-on-year volume', fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(
        lambda v, _: f'{v/1000:.0f}K' if v >= 1000 else str(int(v))))

    # Panel 3: Monthly % share trend
    trend = build_trend_pivot_portal(div, portal, size, abt_in)
    if not trend.empty:
        for idx, (_, nkrow) in enumerate(summary_df.iterrows()):
            nb = nkrow['New_Bucket']
            nb_trend = trend[trend['New_Bucket'] == nb]
            for yi, yr in enumerate(ALL_YEARS):
                pct_col = f'pct_share_{yr}'
                if pct_col in nb_trend.columns:
                    vals = [nb_trend[nb_trend['month'] == m][pct_col].sum() for m in range(1, 13)]
                    ls = '-' if yi == 0 else '--'
                    ax3.plot(range(1, 13), vals, color=colors[idx], linestyle=ls,
                             linewidth=1.5,
                             label=f"NB{int(nb)}-{yr}" if n_buckets <= 6 else None)
    ax3.set_xlabel('Month')
    ax3.set_ylabel('% share of monthly qty')
    ax3.set_title('Monthly % share trend (— 1st yr, -- 2nd yr)', fontweight='bold')
    ax3.set_xticks(range(1, 13))
    ax3.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=8)
    if n_buckets <= 6 and not trend.empty:
        ax3.legend(fontsize=7, ncol=2)

    # Panel 4: Summary table
    ax4.axis('off')
    table_cols = ['NB', 'Key', 'Price Range', 'Vol%'] + [f'Qty {yr}' for yr in ALL_YEARS]
    table_data = []
    for _, r in summary_df.iterrows():
        row_d = [int(r['New_Bucket']), r['archetype_key'],
                 f"₹{int(r['price_min'])}–{int(r['price_max'])}",
                 f"{r['vol_pct']:.1f}%"]
        for yr in ALL_YEARS:
            row_d.append(f"{int(r.get(f'qty_{yr}', 0)):,}")
        table_data.append(row_d)
    tbl = ax4.table(cellText=table_data, colLabels=table_cols, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.4)
    ax4.set_title('Archetype Summary', fontweight='bold', pad=20)

    fname = f'{out_path}{div}_{portal_str}_{size}_full_analysis.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  PNG saved: {div}_{portal_str}_{size}_full_analysis.png")


def make_full_analysis_png_consolidated(div, size, keys_df_in, abt_in, out_path):
    """
    Stacked multi-panel PNG for (Division, Size) consolidated view.
    Two columns (volume bar + YoY qty) per portal row.
    """
    portals = get_portals(div, size, keys_df_in)
    if not portals:
        return

    n_portals = len(portals)
    fig, axes = plt.subplots(n_portals, 2,
                             figsize=(18, 5 * n_portals),
                             squeeze=False)
    fig.suptitle(f"{CHANNEL} | {div} / {size} | Consolidated ({n_portals} portals)",
                 fontsize=14, fontweight='bold', y=1.01)

    for row_idx, portal in enumerate(portals):
        portal_str = portal_to_str(portal)
        summary_df = build_summary_portal(div, portal, size, mapping, keys_df_in, abt_in)
        ax_vol = axes[row_idx][0]
        ax_yoy = axes[row_idx][1]

        if summary_df.empty:
            ax_vol.set_visible(False)
            ax_yoy.set_visible(False)
            continue

        n_buckets = len(summary_df)
        colors    = plt.cm.tab10(np.linspace(0, 1, max(n_buckets, 10)))
        nb_labels = [f"NB{int(r['New_Bucket'])}" for _, r in summary_df.iterrows()]
        vol_pcts  = summary_df['vol_pct'].values

        bars = ax_vol.barh(nb_labels, vol_pcts, color=colors[:n_buckets])
        ax_vol.set_xlabel('% portal volume')
        ax_vol.set_title(f'{portal_str} — volume share', fontweight='bold')
        for bar, val in zip(bars, vol_pcts):
            ax_vol.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                        f'{val:.1f}%', va='center', fontsize=8)
        ax_vol.set_xlim(0, (max(vol_pcts) * 1.2) if len(vol_pcts) else 1)

        qty_cols = [c for c in summary_df.columns if c.startswith('qty_')]
        x = np.arange(n_buckets)
        width = 0.8 / max(len(qty_cols), 1)
        for j, qc in enumerate(qty_cols):
            yr = qc.replace('qty_', '')
            ax_yoy.bar(x + j * width - (len(qty_cols) - 1) * width / 2,
                       summary_df[qc].values, width, label=yr, alpha=0.8)
        ax_yoy.set_xticks(x)
        ax_yoy.set_xticklabels(nb_labels, fontsize=8)
        ax_yoy.set_ylabel('Units (qty)')
        ax_yoy.set_title(f'{portal_str} — YoY volume', fontweight='bold')
        ax_yoy.legend(fontsize=8)
        ax_yoy.yaxis.set_major_formatter(plt.FuncFormatter(
            lambda v, _: f'{v/1000:.0f}K' if v >= 1000 else str(int(v))))

    plt.tight_layout()
    fname = f'{out_path}{div}_{size}_full_analysis.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  PNG saved: {div}_{size}_full_analysis.png")


print('PNG functions defined.')


PNG functions defined.


In [4]:
# ════════════════════════════════════════════════════════════════════
# MAIN REPORTING LOOP
# Option A: per_portal/   — one file set per Division_Portal_Size
# Option B: consolidated/ — one file set per Division_Size
# ════════════════════════════════════════════════════════════════════

import os

per_portal_dir   = os.path.join(OUT_PATH, 'per_portal', '')
consolidated_dir = os.path.join(OUT_PATH, 'consolidated', '')
os.makedirs(per_portal_dir,   exist_ok=True)
os.makedirs(consolidated_dir, exist_ok=True)

# ── Option A: per_portal ──────────────────────────────────────────────────────────────────────────
portal_segments = (keys_df[['Division', 'Portal', 'Size']]
                   .drop_duplicates()
                   .sort_values(['Division', 'Portal', 'Size'])
                   .values.tolist())

print(f"=== Option A — per_portal/ ===")
print(f"Portal segments: {len(portal_segments)}")
for d, p, s in portal_segments:
    print(f"  {d}_{portal_to_str(p)}_{s}")
print()

all_portal_summaries = []

for div, portal, size in portal_segments:
    portal_str = portal_to_str(portal)
    label      = f"{div}_{portal_str}_{size}"
    print(f"── {label} ────────────────────────────────────────────")

    summary   = build_summary_portal(div, portal, size, mapping, keys_df, abt)
    detail    = build_detail_portal(div, portal, size, abt)
    trend_piv = build_trend_pivot_portal(div, portal, size, abt)
    piv_ready = build_pivot_ready_portal(div, portal, size, abt)

    if summary.empty:
        print(f"  SKIP — no data")
        continue

    k = len(summary)
    print(f"  K={k} | price ₹{summary['price_min'].min()}–{summary['price_max'].max()}")

    summary.to_csv(f'{per_portal_dir}{label}_summary.csv',       index=False)
    detail.to_csv(f'{per_portal_dir}{label}_detail.csv',         index=False)
    trend_piv.to_csv(f'{per_portal_dir}{label}_trend_pivot.csv', index=False)
    if not piv_ready.empty:
        piv_ready.to_excel(f'{per_portal_dir}{label}_pivot_ready.xlsx', engine='openpyxl')
    make_full_analysis_png_portal(div, portal, size, summary, abt, per_portal_dir)

    all_portal_summaries.append(summary)

print()
print('=== Option A complete ===')
print()

# ── Option B: consolidated ──────────────────────────────────────────────────────────────────────────
div_size_segments = (keys_df[['Division', 'Size']]
                     .drop_duplicates()
                     .sort_values(['Division', 'Size'])
                     .values.tolist())

print(f"=== Option B — consolidated/ ===")
print(f"Division×Size segments: {len(div_size_segments)}")
for d, s in div_size_segments:
    print(f"  {d}_{s}")
print()

all_consolidated_summaries = []

for div, size in div_size_segments:
    label = f"{div}_{size}"
    print(f"── {label} ────────────────────────────────────────────")

    summary   = build_summary_consolidated(div, size, mapping, keys_df, abt)
    detail    = build_detail_consolidated(div, size, abt)
    trend_piv = build_trend_pivot_consolidated(div, size, abt)
    piv_ready = build_pivot_ready_consolidated(div, size, abt)

    if summary.empty:
        print(f"  SKIP — no data")
        continue

    portals = get_portals(div, size, keys_df)
    print(f"  {len(portals)} portal(s) | {len(summary)} NB rows")

    summary.to_csv(f'{consolidated_dir}{label}_summary.csv',       index=False)
    detail.to_csv(f'{consolidated_dir}{label}_detail.csv',         index=False)
    trend_piv.to_csv(f'{consolidated_dir}{label}_trend_pivot.csv', index=False)
    if not piv_ready.empty:
        piv_ready.to_excel(f'{consolidated_dir}{label}_pivot_ready.xlsx', engine='openpyxl')
    make_full_analysis_png_consolidated(div, size, keys_df, abt, consolidated_dir)

    all_consolidated_summaries.append(summary)

print()
print('=== Option B complete ===')


=== Option A — per_portal/ ===
Portal segments: 14
  BP_TT_Single
  BS_TT_Single
  DF_TT_DF
  DF_TT_DFT
  HL_TT_CABIN
  HL_TT_LARGE
  HL_TT_MEDIUM
  HL_TT_SO2
  HL_TT_SO3
  SL_TT_CABIN
  SL_TT_LARGE
  SL_TT_MEDIUM
  SL_TT_SO2
  SL_TT_SO3

── BP_TT_Single ────────────────────────────────────────────


  K=7 | price ₹0–29499

  PNG saved: BP_TT_Single_full_analysis.png
── BS_TT_Single ────────────────────────────────────────────
  K=4 | price ₹0–6399


  PNG saved: BS_TT_Single_full_analysis.png
── DF_TT_DF ────────────────────────────────────────────
  K=3 | price ₹0–8199


  PNG saved: DF_TT_DF_full_analysis.png
── DF_TT_DFT ────────────────────────────────────────────
  K=5 | price ₹0–6599


...

  1 portal(s) | 8 NB rows


  PNG saved: SL_LARGE_full_analysis.png
── SL_MEDIUM ────────────────────────────────────────────
  1 portal(s) | 8 NB rows


  PNG saved: SL_MEDIUM_full_analysis.png
── SL_SO2 ────────────────────────────────────────────
  1 portal(s) | 7 NB rows


  PNG saved: SL_SO2_full_analysis.png
── SL_SO3 ────────────────────────────────────────────
  1 portal(s) | 8 NB rows


  PNG saved: SL_SO3_full_analysis.png

=== Option B complete ===


In [5]:
# ════════════════════════════════════════════════════════════════════
# MASTER SUMMARY FILES
# per_portal/ALL_SEGMENTS_master_summary.csv    — Portal column, all segs stacked
# consolidated/ALL_DIVISIONS_master_summary.csv — all Div×Size consolidated rows
# ════════════════════════════════════════════════════════════════════

# ── per_portal master ───────────────────────────────────────────────────────────────────────────────
if all_portal_summaries:
    master_portal = pd.concat(all_portal_summaries, ignore_index=True)
    master_portal_sorted = master_portal.sort_values(['Division', 'Portal', 'Size', 'New_Bucket'])
    master_portal_sorted.to_csv(f'{per_portal_dir}ALL_SEGMENTS_master_summary.csv', index=False)
    print(f"Saved per_portal/ALL_SEGMENTS_master_summary.csv  ({len(master_portal_sorted)} rows)")

    print()
    print(f"{'Segment':<30} {'Portals':>8} {'K (total)':>10}  {'Price range':>22}")
    print('─' * 76)
    for (div, size), grp in master_portal.groupby(['Division', 'Size']):
        n_portals = grp['Portal'].nunique()
        k_total   = len(grp)
        mn = grp['price_min'].min()
        mx = grp['price_max'].max()
        print(f"  {div}_{size:<27} {n_portals:>8} {k_total:>10}  ₹{mn:>8,}–{mx:>8,}")
else:
    print('No per-portal summaries generated.')

print()

# ── consolidated master ─────────────────────────────────────────────────────────────────────────────
if all_consolidated_summaries:
    master_cons = pd.concat(all_consolidated_summaries, ignore_index=True)
    master_cons_sorted = master_cons.sort_values(['Division', 'Size', 'New_Bucket'])
    master_cons_sorted.to_csv(f'{consolidated_dir}ALL_DIVISIONS_master_summary.csv', index=False)
    print(f"Saved consolidated/ALL_DIVISIONS_master_summary.csv ({len(master_cons_sorted)} rows)")

    print()
    print(f"{'Segment':<30} {'NB rows':>8}")
    print('─' * 42)
    for (div, size), grp in master_cons.groupby(['Division', 'Size']):
        print(f"  {div}_{size:<27} {len(grp):>8}")
else:
    print('No consolidated summaries generated.')


Saved per_portal/ALL_SEGMENTS_master_summary.csv  (96 rows)

Segment                         Portals  K (total)             Price range
────────────────────────────────────────────────────────────────────────────
  BP_Single                             1          7  ₹       0–  29,499
  BS_Single                             1          4  ₹       0–   6,399
  DF_DF                                 1          3  ₹       0–   8,199
  DF_DFT                                1          5  ₹       0–   6,599
  HL_CABIN                              1          8  ₹       0–   8,399
  HL_LARGE                              1          8  ₹       0–  12,399
  HL_MEDIUM                             1          8  ₹       0–  10,299
  HL_SO2                                1          6  ₹       0–  34,799
  HL_SO3                                1          8  ₹       0–  27,899
  SL_CABIN                              1          8  ₹       0–   8,699
  SL_LARGE                              1          8  ₹  